# Name That LLM — independent Project 2 solution

Attach the competition data, disable internet, and run all cells. This notebook rebuilds the selected model on all labeled rows and predicts the actual hidden test file. The selection comes from five validation folds grouped by prompt, with a separate 30-prompt holdout.

This implementation was written from scratch without research-paper or competitor-notebook code. Predictions do not depend on which other passages appear in the test batch.

**CPU version:** no pretrained checkpoint input is needed.


In [ ]:
import os, sys, json, time
from pathlib import Path
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
WORK=Path(os.environ.get('NTL_WORK_DIR','/kaggle/working'))
WORK.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(WORK))
if os.environ.get('NTL_DATA_DIR'):
    DATA=Path(os.environ['NTL_DATA_DIR'])
else:
    candidates=[p.parent for p in Path('/kaggle/input').rglob('sample_submission.parquet')
                if (p.parent/'train.csv').exists() and (p.parent/'test.csv').exists()]
    if len(candidates)!=1:
        raise RuntimeError(f'Expected one competition dataset, found {candidates}')
    DATA=candidates[0]
print('Data:',DATA)
started=time.perf_counter()


In [ ]:
(WORK/'dataio.py').write_text('"""Dataset contracts, immutable split assignment, and submission validation."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import GroupKFold\n\nSEED = 71423\n\n\ndef digest_file(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef text_hashes(texts):\n    return np.array([hashlib.sha256(t.encode("utf-8")).hexdigest() for t in texts])\n\n\ndef read_competition(directory):\n    directory = Path(directory)\n    train = pd.read_csv(directory/"train.csv", dtype=str)\n    test = pd.read_csv(directory/"test.csv", dtype=str)\n    expected = {"train": ["id", "prompt_id", "text", "model"], "test": ["id", "text"]}\n    for name, frame in [("train", train), ("test", test)]:\n        if frame.columns.tolist() != expected[name]:\n            raise ValueError(f"{name}: unexpected columns")\n        validate_text(frame)\n        if frame.isna().any().any():\n            raise ValueError(f"{name}: missing cell")\n    labels = sorted(train.model.unique())\n    if len(labels) != 18 or train.prompt_id.nunique() < 40:\n        raise ValueError("Unexpected competition training data")\n    if set(train.id) & set(test.id):\n        raise ValueError("Train/test IDs overlap")\n    sample = pd.read_parquet(directory/"sample_submission.parquet")\n    if sample.columns.tolist() != ["id", "model"]:\n        raise ValueError("Unexpected sample submission schema")\n    # Only check the sample\'s format: hidden test files may have different IDs.\n    return train, test, labels\n\n\ndef validate_text(frame):\n    if not {"id", "text"}.issubset(frame.columns):\n        raise ValueError("Input requires id and text")\n    if frame[["id", "text"]].isna().any().any() or not frame.id.is_unique:\n        raise ValueError("Missing text/ID or duplicate IDs")\n    if frame.id.str.strip().eq("").any() or frame.text.str.strip().eq("").any():\n        raise ValueError("Empty text/ID")\n\n\ndef split_prompts(train, destination):\n    """30 holdout prompts; five folds across all remaining prompts."""\n    destination = Path(destination)\n    if destination.exists():\n        saved = pd.read_csv(destination, dtype={"id": str, "prompt_id": str})\n        if not saved[["id", "prompt_id"]].equals(train[["id", "prompt_id"]]):\n            raise ValueError("Saved split belongs to different data")\n        if saved.groupby("prompt_id").fold.nunique().max() != 1:\n            raise ValueError("A prompt crosses split boundaries")\n        if set(saved.fold) != {-1, 0, 1, 2, 3, 4}:\n            raise ValueError("Invalid fold assignments")\n        if saved.loc[saved.fold == -1, "prompt_id"].nunique() != 30:\n            raise ValueError("Expected 30 holdout prompts")\n        return saved\n    ordered = sorted(train.prompt_id.unique(),\n                     key=lambda g: hashlib.sha256(f"{SEED}:{g}".encode()).hexdigest())\n    held = set(ordered[:30])\n    assigned = train[["id", "prompt_id"]].copy()\n    assigned["fold"] = -1\n    indices = np.flatnonzero(~train.prompt_id.isin(held))\n    cv = GroupKFold(n_splits=5, shuffle=True, random_state=SEED)\n    for fold, (_, val) in enumerate(cv.split(indices, groups=train.prompt_id.iloc[indices])):\n        assigned.loc[indices[val], "fold"] = fold\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    assigned.to_csv(destination, index=False)\n    return assigned\n\n\ndef embedding_rows(cache, frame):\n    if cache is None:\n        return None\n    with np.load(cache, allow_pickle=False) as f:\n        lookup = {row_id: i for i, row_id in enumerate(f["ids"])}\n        if len(lookup) != len(f["ids"]):\n            raise ValueError("Duplicate embedding IDs")\n        positions = [lookup[row_id] for row_id in frame.id]\n        if not np.array_equal(f["hashes"][positions], text_hashes(frame.text)):\n            raise ValueError("Embedding text checksum mismatch")\n        result = f["vectors"][positions]\n    if not np.isfinite(result).all():\n        raise ValueError("Nonfinite embeddings")\n    return result\n\n\ndef embedding_spec(cache):\n    if cache is None:\n        return None\n    with np.load(cache, allow_pickle=False) as f:\n        return json.loads(str(f["specification"]))\n\n\ndef submit(test, probabilities, labels, path):\n    validate_text(test)\n    probabilities = np.asarray(probabilities)\n    if probabilities.shape != (len(test), len(labels)):\n        raise ValueError("Prediction shape mismatch")\n    if not np.isfinite(probabilities).all() or (probabilities < 0).any():\n        raise ValueError("Invalid prediction values")\n    if not np.allclose(probabilities.sum(axis=1), 1, atol=1e-5):\n        raise ValueError("Prediction rows must sum to one")\n    result = pd.DataFrame({\n        "id": test.id.to_numpy(),\n        "model": np.asarray(labels)[probabilities.argmax(axis=1)],\n    })\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    result.to_parquet(path, index=False)\n    pd.testing.assert_frame_equal(pd.read_parquet(path), result)\n    return result\n')


In [ ]:
(WORK/'attribution.py').write_text('"""Original text attribution implementation, developed from dataset inspection.\n\nEvery prediction depends only on that passage and the fitted training model.\nChanging test order, batch size, or other test passages cannot change it.\n"""\nfrom __future__ import annotations\n\nimport re\nimport time\nfrom collections import Counter\n\nimport numpy as np\nfrom catboost import CatBoostClassifier\nfrom scipy import sparse\nfrom scipy.special import softmax\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS\nfrom sklearn.linear_model import LogisticRegression, RidgeClassifier, Ridge\nfrom sklearn.preprocessing import StandardScaler, RobustScaler\nfrom sklearn.svm import LinearSVC\n\nfrom dataio import SEED\n\nVERSION = "project2-1"\nWORD = re.compile(r"[^\\W\\d_]+(?:[\'’][^\\W\\d_]+)?", re.UNICODE)\nGLUE = tuple(sorted({\n    "a", "an", "the", "and", "or", "but", "yet", "so", "as", "if", "than",\n    "of", "to", "for", "in", "on", "at", "by", "with", "from", "into", "within",\n    "without", "through", "between", "over", "under", "while", "where", "when",\n    "which", "that", "this", "these", "those", "it", "its", "they", "their",\n    "we", "our", "you", "your", "is", "are", "was", "were", "be", "been",\n    "can", "could", "would", "may", "might", "will", "must", "not", "no",\n    "only", "also", "even", "rather", "just", "more", "less", "each", "every",\n    "both", "all", "because", "however", "therefore", "thus", "instead",\n    "still", "often", "sometimes", "ultimately", "merely",\n}))\nPUNCTUATION = tuple(".,;:!?-\'’\\"“”()—–/…")\n\n\ndef grammatical_outline(text):\n    def replace(match):\n        word = match.group()\n        if word.lower() in ENGLISH_STOP_WORDS:\n            return word.lower()\n        size = len(word)\n        return ("CAP" if word[0].isupper() else "word") + str(min(size//3, 4))\n    return WORD.sub(replace, text)\n\n\ndef numeric_features(texts):\n    rows = []\n    for text in texts:\n        tokens = WORD.findall(text)\n        n = max(1, len(tokens))\n        size = max(1, len(text))\n        vocabulary = Counter(t.lower() for t in tokens)\n        lengths = np.array([len(t) for t in tokens] or [0])\n        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\\s+", text) if s.strip()]\n        sl = np.array([len(WORD.findall(s)) for s in sentences] or [0])\n        frequency = np.array(list(vocabulary.values()) or [1])\n        row = [\n            np.log1p(size), np.log1p(n), size/n, len(vocabulary)/n,\n            (frequency == 1).sum()/n, frequency.max()/n,\n            (frequency*(frequency-1)).sum()/max(n*(n-1), 1),\n            np.mean(lengths), np.std(lengths), np.median(lengths),\n            np.quantile(lengths, .9), np.log1p(len(sentences)),\n            np.mean(sl), np.std(sl), np.median(sl), sl.max(),\n            sum(t[0].isupper() for t in tokens)/n,\n            sum(t.isupper() and len(t)>1 for t in tokens)/n,\n            sum(t.lower() in ENGLISH_STOP_WORDS for t in tokens)/n,\n            sum(c.isdigit() for c in text)/size,\n            sum(ord(c)>127 for c in text)/size,\n        ]\n        row.extend(np.mean(lengths == length) for length in range(1, 13))\n        row.extend(vocabulary[word]/n for word in GLUE)\n        for symbol in PUNCTUATION:\n            count = text.count(symbol)\n            row.extend([count/n, np.log1p(count)])\n        # Separate punctuation habits near openings and endings.\n        for segment in [text[:80], text[-80:]]:\n            row.extend(segment.count(p)/max(1, len(segment)) for p in [",", ":", ";", "—", "’", "?"])\n        # Sentence-level structural markers and repetition.\n        beginnings = [WORD.findall(s.lower())[:1] for s in sentences]\n        row.extend(sum(bool(b) and b[0] == w for b in beginnings)/max(1,len(sentences))\n                   for w in ["the", "a", "it", "this", "yet", "but", "in", "as", "we"])\n        row.extend([\n            len(re.findall(r"\\w—\\w", text))/n,\n            len(re.findall(r"\\s—\\s", text))/n,\n            len(re.findall(r"\\w-\\w", text))/n,\n            len(re.findall(r"\\w[\'’]\\w", text))/n,\n            float(text.rstrip().endswith(".")),\n            float(text.rstrip().endswith("?")),\n            float(text.rstrip().endswith(\'"\')),\n        ])\n        rows.append(row)\n    return np.asarray(rows, dtype=np.float32)\n\n\nclass Representations:\n    def fit(self, texts, vectors=None):\n        self.char = TfidfVectorizer(\n            analyzer="char", ngram_range=(2, 6), min_df=3, max_features=130_000,\n            lowercase=False, sublinear_tf=True, dtype=np.float32)\n        self.word = TfidfVectorizer(\n            ngram_range=(1, 2), min_df=2, max_features=50_000,\n            token_pattern=r"(?u)\\b[\\w’\']+\\b", sublinear_tf=True, dtype=np.float32)\n        self.outline = TfidfVectorizer(\n            analyzer="char", ngram_range=(3, 6), min_df=3, max_features=35_000,\n            lowercase=False, sublinear_tf=True, dtype=np.float32)\n        self.char.fit(texts)\n        self.word.fit(texts)\n        self.outline.fit([grammatical_outline(t) for t in texts])\n        self.scaler = RobustScaler(quantile_range=(10, 90)).fit(numeric_features(texts))\n        self.vector_scaler = None if vectors is None else StandardScaler().fit(vectors)\n        return self\n\n    def transform(self, texts, vectors=None):\n        lexical = sparse.hstack([self.char.transform(texts), .65*self.word.transform(texts)], format="csr")\n        numbers = numeric_features(texts)\n        scaled = np.clip(self.scaler.transform(numbers), -12, 12).astype(np.float32)\n        combined = sparse.hstack([\n            lexical, .7*self.outline.transform([grammatical_outline(t) for t in texts]),\n            sparse.csr_matrix(.06*scaled),\n        ], format="csr")\n        result = {"lexical": lexical, "combined": combined, "numbers": numbers, "scaled": scaled}\n        if self.vector_scaler is not None:\n            if vectors is None:\n                raise ValueError("This model requires encoder vectors")\n            result["vectors"] = self.vector_scaler.transform(vectors).astype(np.float32)\n            result["vector_style"] = np.hstack([result["vectors"], scaled*2])\n        return result\n\n\nclass PromptRegularizedRidge:\n    """Discourage logits from tracking training prompt means.\n\n    Extra rows represent the average completion for each training prompt.\n    Their target is the same uniform class distribution. No prompt information\n    or other test passages are needed when predicting.\n    """\n    requires_groups = True\n\n    def __init__(self,alpha=1000,strength=5):\n        self.alpha,self.strength=alpha,strength\n\n    def fit(self,x,y,groups):\n        if groups is None:\n            raise ValueError("Training prompt IDs are required for this component")\n        self.classes_=np.unique(y)\n        target=np.eye(len(self.classes_))[y]-1/len(self.classes_)\n        nuisance,targets=[],[]\n        for group in np.unique(groups):\n            rows=np.flatnonzero(groups==group)\n            nuisance.append(x[rows].mean(0)*np.sqrt(len(rows)*self.strength))\n            targets.append(np.zeros(len(self.classes_)))\n        self.estimator=Ridge(alpha=self.alpha,fit_intercept=False,solver="lsqr")\n        self.estimator.fit(np.vstack([x,nuisance]),np.vstack([target,targets]))\n        return self\n\n    def predict_proba(self,x):\n        return softmax(self.estimator.predict(x)*8,axis=1)\n\n\ndef definitions(use_encoder):\n    specs = {\n        "lexical_svm": ("lexical", LinearSVC(C=1.0, dual=True, max_iter=5000,\n                                            class_weight="balanced", random_state=SEED)),\n        "structure_svm": ("combined", LinearSVC(C=1.0, dual=True, max_iter=5000,\n                                               class_weight="balanced", random_state=SEED)),\n        "structure_logistic": ("combined", LogisticRegression(C=4.0, max_iter=1500,\n                                                              class_weight="balanced")),\n        "style_boost": ("numbers", CatBoostClassifier(\n            iterations=650, depth=5, learning_rate=.045, l2_leaf_reg=8,\n            loss_function="MultiClass", random_seed=SEED, thread_count=4,\n            verbose=False, allow_writing_files=False)),\n        "style_forest": ("numbers", ExtraTreesClassifier(\n            n_estimators=600, max_features=.6, min_samples_leaf=3,\n            class_weight="balanced", n_jobs=4, random_state=SEED)),\n    }\n    if use_encoder:\n        specs.update({\n            "encoder_ridge": ("vectors", RidgeClassifier(alpha=100, solver="lsqr", class_weight="balanced")),\n            "encoder_regularized": ("vector_style", RidgeClassifier(alpha=1000, solver="lsqr", class_weight="balanced")),\n            "encoder_logistic": ("vector_style", LogisticRegression(C=.015, max_iter=1500,\n                                                                    class_weight="balanced")),\n            "encoder_logistic_strong": ("vector_style", LogisticRegression(C=.004, max_iter=1500,\n                                                                           class_weight="balanced")),\n            "encoder_logistic_light": ("vector_style", LogisticRegression(C=.06, max_iter=1500,\n                                                                          class_weight="balanced")),\n            "prompt_regularized": ("vector_style", PromptRegularizedRidge(alpha=1000,strength=5)),\n            "prompt_regularized_light": ("vector_style", PromptRegularizedRidge(alpha=300,strength=5)),\n        })\n    return specs\n\n\nclass Predictor:\n    def __init__(self, labels, selected=None):\n        self.labels = list(labels)\n        self.selected = selected\n\n    def fit(self, texts, y, vectors=None, groups=None):\n        self.features = Representations().fit(texts, vectors)\n        matrices = self.features.transform(texts, vectors)\n        self.members = {}\n        self.neural_members = {}\n        for name, (view, estimator) in definitions(vectors is not None).items():\n            if self.selected is not None and name not in self.selected:\n                continue\n            started = time.perf_counter()\n            if getattr(estimator,"requires_groups",False):\n                estimator.fit(matrices[view], y, groups)\n            else:\n                estimator.fit(matrices[view], y)\n            self.members[name] = (view, estimator)\n            print(f"  {name}: {time.perf_counter()-started:.1f}s", flush=True)\n        for name in self.selected or []:\n            if name.startswith("neural_e"):\n                from neural import NeuralModel\n                self.neural_members[name] = NeuralModel(\n                    len(self.labels), epochs=int(name.removeprefix("neural_e"))).fit(texts, y)\n        return self\n\n    def predict_members(self, texts, vectors=None):\n        matrices = self.features.transform(texts, vectors)\n        output = {}\n        for name, (view, estimator) in self.members.items():\n            if hasattr(estimator, "predict_proba"):\n                p = estimator.predict_proba(matrices[view])\n            else:\n                score = estimator.decision_function(matrices[view])\n                if score.ndim == 1:\n                    score = np.column_stack([-score, score])\n                # Fixed score temperatures; weights are selected using OOF data.\n                temperature = 5.0 if "svm" in name else 4.0\n                p = softmax(score*temperature, axis=1)\n            aligned = np.zeros((len(texts), len(self.labels)), dtype=np.float64)\n            aligned[:, np.asarray(estimator.classes_, dtype=int)] = p\n            output[name] = aligned.astype(np.float32)\n        for name, model in self.neural_members.items():\n            output[name] = model.predict_proba(texts)\n        return output\n\n\ndef weighted_predictions(members, weights):\n    if any(v < 0 for v in weights.values()) or sum(weights.values()) <= 0:\n        raise ValueError("Invalid ensemble weights")\n    prediction = sum(members[name]*weight for name,weight in weights.items()) / sum(weights.values())\n    return prediction/prediction.sum(axis=1, keepdims=True)\n')


In [ ]:
import pandas as pd
from threadpoolctl import threadpool_limits
from dataio import read_competition, digest_file, submit, embedding_rows, embedding_spec
from attribution import Predictor, weighted_predictions, VERSION
selection=json.loads("{\"specification\": {\"version\": \"project2-1\", \"seed\": 71423, \"train_sha256\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\", \"encoder\": null, \"labels\": [\"claude-fable-5\", \"claude-opus-4-6\", \"claude-opus-4-8\", \"claude-opus-5\", \"claude-sonnet-5\", \"gemini-3.1-pro-preview\", \"gemini-3.6-flash\", \"gemini-3.7-flash\", \"gemini-3.8-flash\", \"glm-5.3\", \"glm-5.3-flash\", \"gpt-5.5\", \"gpt-5.6-luna\", \"gpt-5.6-sol\", \"gpt-5.6-terra\", \"gpt-6-astra\", \"kimi-k3\", \"qwen3.8-max\"]}, \"development_rows\": 2159, \"holdout_rows\": 540, \"component_scores\": {\"lexical_svm\": 0.31484593837535013, \"structure_svm\": 0.36623482726423906, \"structure_logistic\": 0.36626595082477437, \"style_boost\": 0.3102513227513228, \"style_forest\": 0.279703548085901}, \"selected\": {\"weights\": {\"structure_logistic\": 0.3333333333333333, \"lexical_svm\": 0.3333333333333333, \"style_boost\": 0.3333333333333333}, \"score\": 0.3796840958605665, \"fold_scores\": [0.3611111111111111, 0.3843599033816425, 0.4027777777777778, 0.3842592592592593, 0.3657407407407407]}, \"candidate_count\": 51, \"top_candidates\": [{\"weights\": {\"structure_logistic\": 0.3333333333333333, \"lexical_svm\": 0.3333333333333333, \"style_boost\": 0.3333333333333333}, \"score\": 0.3796840958605665, \"fold_scores\": [0.3611111111111111, 0.3843599033816425, 0.4027777777777778, 0.3842592592592593, 0.3657407407407407]}, {\"weights\": {\"structure_logistic\": 0.3333333333333333, \"structure_svm\": 0.3333333333333333, \"style_boost\": 0.3333333333333333}, \"score\": 0.37968409586056645, \"fold_scores\": [0.3726851851851852, 0.37037037037037035, 0.40740740740740744, 0.3819444444444444, 0.36574074074074076]}, {\"weights\": {\"structure_svm\": 0.3333333333333333, \"lexical_svm\": 0.3333333333333333, \"style_boost\": 0.3333333333333333}, \"score\": 0.3792172424525366, \"fold_scores\": [0.3726851851851851, 0.3657407407407407, 0.4050925925925926, 0.38888888888888895, 0.36342592592592593]}, {\"weights\": {\"structure_logistic\": 0.75, \"style_boost\": 0.25}, \"score\": 0.37738873327108624, \"fold_scores\": [0.3657407407407407, 0.3797302737520129, 0.4050925925925926, 0.37731481481481477, 0.35879629629629634]}, {\"weights\": {\"structure_logistic\": 0.25, \"structure_svm\": 0.25, \"style_boost\": 0.5}, \"score\": 0.3769102085278556, \"fold_scores\": [0.36342592592592593, 0.3843599033816425, 0.412037037037037, 0.3842592592592593, 0.3402777777777778]}, {\"weights\": {\"structure_logistic\": 0.5, \"structure_svm\": 0.25, \"style_boost\": 0.25}, \"score\": 0.37690242763772175, \"fold_scores\": [0.3703703703703704, 0.36342592592592593, 0.40972222222222227, 0.3773148148148148, 0.36342592592592593]}, {\"weights\": {\"structure_logistic\": 0.75, \"style_forest\": 0.25}, \"score\": 0.37646669779022723, \"fold_scores\": [0.3773148148148148, 0.3867753623188406, 0.40046296296296297, 0.3564814814814815, 0.3611111111111111]}, {\"weights\": {\"structure_svm\": 0.5, \"style_forest\": 0.5}, \"score\": 0.37645113600995955, \"fold_scores\": [0.3634259259259259, 0.36584138486312395, 0.41203703703703703, 0.3657407407407407, 0.375]}, {\"weights\": {\"structure_svm\": 0.5, \"style_boost\": 0.5}, \"score\": 0.37643557422969187, \"fold_scores\": [0.3634259259259259, 0.3703703703703704, 0.40740740740740744, 0.3796296296296296, 0.3611111111111111]}, {\"weights\": {\"structure_logistic\": 0.25, \"structure_svm\": 0.5, \"style_boost\": 0.25}, \"score\": 0.37505057578586987, \"fold_scores\": [0.37499999999999994, 0.37037037037037035, 0.3912037037037037, 0.375, 0.3634259259259259]}], \"fold_seconds\": [55.81690987502225, 56.9586208750261, 60.40639112505596, 62.16453641699627, 55.76499062497169], \"note\": \"Tuned development CV. Holdout has not been evaluated by this command.\"}")
spec=selection['specification']
assert spec['version']==VERSION
if digest_file(DATA/'train.csv')!=spec['train_sha256']:
    raise RuntimeError('Training data changed. Revalidate before submitting.')
train,test,labels=read_competition(DATA)
assert labels==spec['labels']
y=pd.Categorical(train.model,categories=labels).codes
print('Training rows:',len(train),'Runtime test rows:',len(test))


In [ ]:
training_vectors=test_vectors=None


In [ ]:
weights=selection['selected']['weights']
with threadpool_limits(limits=4):
    predictor=Predictor(labels,selected=weights).fit(train.text.tolist(),y,training_vectors,train.prompt_id.to_numpy())
    scores=weighted_predictions(predictor.predict_members(test.text.tolist(),test_vectors),weights)
submission=submit(test,scores,labels,WORK/'submission.parquet')
assert submission.id.tolist()==test.id.tolist()
print(submission.head())
print(f'Wrote {len(submission)} predictions in {(time.perf_counter()-started)/60:.1f} minutes')
